# PDHG FFHQ-100 Local Refinement In Colab

This notebook runs a local refinement sweep over a 100-image FFHQ slice.

Default design:
- `total_images=100`
- `batch_size=100`
- fixed sigma schedule at `sigma_max=27`, `sigma_min=0.075`
- 3 local `tau` values
- 3 local `sigma_dual` values
- cross-product = 9 candidates total
- same seed as the coarse sweep (`seed=99`)

It is meant to refine the current best region rather than rescan the full coarse grid.

## Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

Prefer `A100` or `H100`. If `batch_size=100` is too large for the runtime, reduce it to `50` or `25` and rerun.

In [ ]:
#@title Project Settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "codex-pdhg-colab-light-100"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_tuning_exports"  #@param {type:"string"}
DRIVE_FFHQ_DATA_DIR = "/content/drive/MyDrive/mycode/test-ffhq"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}

SEED = 99  #@param {type:"integer"}
TOTAL_IMAGES = 100  #@param {type:"integer"}
BATCH_SIZE = 100  #@param {type:"integer"}
DATA_START_IDX = 0  #@param {type:"integer"}
NUM_STEPS = 500  #@param {type:"integer"}
MAX_ITER = 500  #@param {type:"integer"}
FFHQ_MEASUREMENT_SIGMA = 0.05  #@param {type:"number"}
PRIMARY_METRIC = "psnr"  #@param ["psnr", "ssim", "lpips"]
LIGHT_METRICS_ONLY = False  #@param {type:"boolean"}
MAX_CANDIDATES = 0  #@param {type:"integer"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}

SIGMA_MAX = 27.0  #@param {type:"number"}
SIGMA_MIN = 0.075  #@param {type:"number"}

# Three local tau values separated by semicolons.
TAU_LIST = "0.009;0.01;0.011"  #@param {type:"string"}

# Three local sigma_dual values separated by semicolons.
SIGMA_DUAL_LIST = "1400;1600;1800"  #@param {type:"string"}


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Fetch The Repo
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            repo_dir.as_posix(),
        ],
        check=True,
    )
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(repo_dir.parent)
    extracted_root = repo_dir.parent / zip_path.stem
    if extracted_root.exists() and extracted_root != repo_dir:
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        extracted_root.rename(repo_dir)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(repo_dir)
print(f"Repo ready: {repo_dir}")


In [ ]:
#@title Install Colab Dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Installed requirements-colab.txt")


In [ ]:
#@title Download The FFHQ Checkpoint If Needed
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
ckpt_path = Path("pretrained-models/ffhq_10m.pt")
ckpt_path.parent.mkdir(parents=True, exist_ok=True)

if ckpt_path.exists():
    print(f"Checkpoint already present: {ckpt_path}")
else:
    subprocess.run(
        [
            "gdown",
            "--id",
            "1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh",
            "-O",
            ckpt_path.as_posix(),
        ],
        check=True,
    )
    print(f"Downloaded checkpoint to: {ckpt_path}")


In [ ]:
#@title Write The 9-Candidate Config
import json
import os
import shlex
import time
from pathlib import Path

import yaml

os.chdir(REPO_DIR)
repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_FFHQ_DATA_DIR)
if not drive_data_dir.exists():
    raise FileNotFoundError(f"FFHQ dataset path not found: {drive_data_dir}")

def parse_float_list(text: str, label: str):
    items = []
    for raw_item in text.split(";"):
        raw_item = raw_item.strip()
        if not raw_item:
            continue
        items.append(float(raw_item))
    if not items:
        raise ValueError(f"No valid {label} values found in: {text!r}")
    return items

session_tag = SESSION_TAG.strip() or time.strftime("%Y%m%d-%H%M%S")
study_name = f"pdhg_phase_retrieval_colab_refine_{session_tag}"
config_path = repo_dir / "tuning" / f"pdhg_exact_list.colab_{session_tag}.yaml"
study_root = repo_dir / "tuning_runs" / study_name
results_root = repo_dir / "results" / "tuning" / study_name
outputs_root = repo_dir / "outputs" / "tuning" / study_name
latest_log_path = repo_dir / "tuning_runs" / f"latest_refine_100_{session_tag}.log"
latest_pid_path = repo_dir / "tuning_runs" / f"latest_refine_100_{session_tag}.pid"
data_end_idx = DATA_START_IDX + TOTAL_IMAGES

tau_values = parse_float_list(TAU_LIST, "tau")
sigma_dual_values = parse_float_list(SIGMA_DUAL_LIST, "sigma_dual")

candidates = []
for tau_index, tau in enumerate(tau_values, start=1):
    for sigma_dual_index, sigma_dual in enumerate(sigma_dual_values, start=1):
        candidates.append(
            {
                "name": f"tau_{tau_index}_dual_{sigma_dual_index}",
                "params": {
                    "sampler.annealing_scheduler_config.sigma_max": SIGMA_MAX,
                    "sampler.annealing_scheduler_config.sigma_min": SIGMA_MIN,
                    "inverse_task.admm_config.pdhg.tau": tau,
                    "inverse_task.admm_config.pdhg.sigma_dual": sigma_dual,
                },
            }
        )

eval_fn_override = "eval_fn_list=[psnr]" if LIGHT_METRICS_ONLY else "eval_fn_list=[psnr,ssim,lpips]"

cfg = {
    "study": {
        "name": study_name,
        "output_root": (repo_dir / "tuning_runs").as_posix(),
        "result_root": (repo_dir / "results" / "tuning").as_posix(),
        "hydra_root": (repo_dir / "outputs" / "tuning").as_posix(),
    },
    "runner": {
        "python_executable": PYTHON_BIN,
        "entrypoint": "recover_inverse2.py",
        "config_name": "default_ffhq.yaml",
        "workdir": repo_dir.as_posix(),
    },
    "scoring": {
        "primary_metric": PRIMARY_METRIC,
        "comparator": "auto",
        "aggregate": "mean",
    },
    "report": {
        "top_k": 9,
    },
    "base_overrides": [
        "sampler=edm_pdhg",
        "inverse_task=phase_retrieval",
        f"name=PDHG_Refine_100_{session_tag}",
        f"seed={SEED}",
        "gpu=0",
        "wandb=false",
        "save_samples=false",
        "save_traj=false",
        "save_traj_raw_data=false",
        "show_config=false",
        f"total_images={TOTAL_IMAGES}",
        f"batch_size={BATCH_SIZE}",
        "num_runs=1",
        f"inverse_task.operator.sigma={FFHQ_MEASUREMENT_SIGMA}",
        f"sampler.annealing_scheduler_config.num_steps={NUM_STEPS}",
        f"inverse_task.admm_config.max_iter={MAX_ITER}",
        "inverse_task.admm_config.denoise.final_step=tweedie",
        "inverse_task.admm_config.dys.gamma=0.0075",
        "inverse_task.admm_config.dys.lambda_schedule.activate=true",
        "inverse_task.admm_config.dys.lambda_schedule.start=1",
        "inverse_task.admm_config.dys.lambda_schedule.end=1",
        "inverse_task.admm_config.dys.lambda_schedule.warmup=0",
        "inverse_task.admm_config.denoise.lgvd.num_steps=0",
        eval_fn_override,
        f"data.image_root_path={drive_data_dir.as_posix()}",
        f"data.start_idx={DATA_START_IDX}",
        f"data.end_idx={data_end_idx}",
    ],
    "candidates": candidates,
}

config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

run_script = repo_dir / "tuning" / "run_pdhg_exact_list.py"
launch_cmd = [PYTHON_BIN, run_script.as_posix(), "--config", config_path.as_posix()]
if MAX_CANDIDATES > 0:
    launch_cmd.extend(["--max-candidates", str(MAX_CANDIDATES)])

print(f"Config written to: {config_path}")
print(f"Candidate count: {len(candidates)}")
print(f"Study root: {study_root}")
print(f"Dataset slice: [{DATA_START_IDX}, {data_end_idx})")
print("\nLaunch command:")
print(" ".join(shlex.quote(part) for part in launch_cmd))
print("\nRendered config:\n")
print(yaml.safe_dump(cfg, sort_keys=False))

last_context = {
    "study_name": study_name,
    "study_root": study_root.as_posix(),
    "results_root": results_root.as_posix(),
    "outputs_root": outputs_root.as_posix(),
    "config_path": config_path.as_posix(),
    "launch_cmd": launch_cmd,
    "latest_log_path": latest_log_path.as_posix(),
    "latest_pid_path": latest_pid_path.as_posix(),
}
context_path = repo_dir / "tuning" / f"pdhg_refine_100.context_{session_tag}.json"
with context_path.open("w", encoding="utf-8") as handle:
    json.dump(last_context, handle, indent=2)
print(f"Context saved to: {context_path}")


In [ ]:
#@title Dry-Run The Candidate List
import os
import shlex
import subprocess

os.chdir(REPO_DIR)
dry_cmd = list(launch_cmd) + ["--dry-run"]
print(" ".join(shlex.quote(part) for part in dry_cmd))
subprocess.run(dry_cmd, check=True)


In [ ]:
#@title Launch The Sweep In Background
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
latest_log_path = Path(last_context["latest_log_path"])
latest_pid_path = Path(last_context["latest_pid_path"])
latest_log_path.parent.mkdir(parents=True, exist_ok=True)

with latest_log_path.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        launch_cmd,
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

latest_pid_path.write_text(str(process.pid), encoding="utf-8")
print(f"PID: {process.pid}")
print(f"Log: {latest_log_path}")
print(f"Study: {study_root}")


In [ ]:
#@title Show Recent Launcher Log Lines
from pathlib import Path

log_path = Path(last_context["latest_log_path"])
if not log_path.exists():
    raise FileNotFoundError(f"Log file not found: {log_path}")

lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
tail = lines[-int(LOG_TAIL_LINES):]
print("\n".join(tail) if tail else "<log is empty>")


In [ ]:
#@title Show Progress And Leaderboard
import csv
import json
from pathlib import Path

study_root = Path(last_context["study_root"])
progress_path = study_root / "progress.json"
leaderboard_path = study_root / "leaderboard.csv"

if progress_path.exists():
    print("progress.json:\n")
    print(progress_path.read_text(encoding="utf-8"))
else:
    print(f"Progress file not written yet: {progress_path}")

print("\nleaderboard.csv:\n")
if leaderboard_path.exists():
    with leaderboard_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        print("Leaderboard is empty.")
    else:
        for rank, row in enumerate(rows[:9], start=1):
            print(
                f"{rank:>2}. idx={row.get('candidate_index')} name={row.get('candidate_name')} status={row.get('status')} "
                f"score={row.get('score')} psnr={row.get('psnr_max_mean', 'n/a')} "
                f"ssim={row.get('ssim_max_mean', 'n/a')} lpips={row.get('lpips_min_mean', 'n/a')} "
                f"sigma_max={row.get('sampler.annealing_scheduler_config.sigma_max')} "
                f"sigma_min={row.get('sampler.annealing_scheduler_config.sigma_min')} "
                f"tau={row.get('inverse_task.admm_config.pdhg.tau')} "
                f"sigma_dual={row.get('inverse_task.admm_config.pdhg.sigma_dual')}"
            )
else:
    print(f"Leaderboard not written yet: {leaderboard_path}")


In [ ]:
#@title Copy Study Artifacts To Drive
import shutil
from pathlib import Path

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)

targets = [
    Path(last_context["study_root"]),
    Path(last_context["results_root"]),
    Path(last_context["outputs_root"]),
    Path(last_context["latest_log_path"]),
]

for src in targets:
    if not src.exists():
        print(f"Skipping missing path: {src}")
        continue

    dst = export_root / src.name
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
